# Single guide analyses

## Table of Contents:

* [0. Dependencies](#Dependencies)
* [1. Preparing the environment](#Preparing-the-environment)
* [2. Prepaing input datasets for singles analyses](#Preparing-input-datasets-for-singles-analyses) 
* [3. Running MAGeCK and BAGEL2](#Running-MAGeCK-and-BAGEL2) 
* [4. Collating single guide results](#Collating-single-guide-results)

***

## Dependencies

Please see [INSTALL_README.md](../INSTALL_README.md) for the installation of software dependencies. This Notebook assumes that [R](https://cran.r-project.org/) and library dependencies have been installed and that the `RScript` command is available with access to the relevant data.

***

## Preparing the environment

Several paths are used as input to more than one script. To simplify the commands and improve usability, commonly used paths are stored as environment variables. Those variables are then used in the script arguments to shorten input and output data paths. 

This Notebook assumes that you have the following directory structure and files in place before running the commands: 

* **Top level directory** (`REPO_PATH`)
    * **DATA**
        * **preprocessing**
            * *count_matrix.scaled.tsv*
            * *lfc_matrix.scaled.tsv*
        * **single_guide/01_MAGeCK** (`MAGECK_PATH`)
        * **single_guide/02_BAGEL2** (`BAGEL2_PATH`)
        * **single_guide/03_postprocessing** (`POST_PATH`)
        * **RDS/single_guide** (`RDS_PATH`)
    * **LOGS** (`LOG_PATH`)
    * **METADATA**
        * *sample_annotations.tsv*
        * *CEGv2.txt*
        * *NEGv1.txt*

The scripts require several files to be present:

* `METADATA/sample_annotations.tsv` - sample metadata.
* `DATA/preprocessing/count_matrix.scaled.tsv` - scaled count matrix generated by `JupyterNotebooks/02_Quality_Control.ipynb`
* `DATA/preprocessing/lfc_matrix.scaled.tsv` - scaled fold change matrix generated by `JupyterNotebooks/02_Quality_Control.ipynb`
* `METADATA/CEGv2.txt` - common essential genes from [BAGEL2](https://github.com/hart-lab/bagel) commit c6af217
* `METADATA/NEGv1.txt` - common non-essential genes from [BAGEL2](https://github.com/hart-lab/bagel) commit c6af217

In [1]:
# Set the top level path for the repository
export REPO_PATH=$(dirname `pwd`)

# Show the repository path
echo "Repository path: ${REPO_PATH}"

# Check the repository path exists (don't need to check subdirectories)
if [ ! -d "${REPO_PATH}" ]; then
  echo "Top level directory path does not exist: ${REPO_PATH}"
fi

Repository path: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper


Once the top level directory has been set (this will likely be the path to your clone of the repository), we then set several other resusable paths as environment variables and create their directories if they don't exist. It should not be assumed this exist when you clone the repository as they may be present in the .gitignore files (e.g. output logs or large data files).

In [2]:
# Set environment variables for reusable paths
export MAGECK_PATH="${REPO_PATH}/DATA/single_guide/01_MAGeCK"
export BAGEL2_PATH="${REPO_PATH}/DATA/single_guide/02_BAGEL2"
export POST_PATH="${REPO_PATH}/DATA/single_guide/03_postprocessing"
export LOG_PATH="${REPO_PATH}/LOGS"
export MAGECK_LOG_PATH="${LOG_PATH}/MAGeCK"
export BAGEL2_LOG_PATH="${LOG_PATH}/BAGEL2"
export RDS_PATH="${REPO_PATH}/DATA/RDS/single_guide"

# Show the paths (debug)
echo "MAGeCK output directory: ${MAGECK_PATH}"
echo "MAGeCK log directory: ${MAGECK_LOG_PATH}"
echo "BAGEL2 output directory: ${BAGEL2_PATH}"
echo "MAGeCK log directory: ${BAGEL2_LOG_PATH}"
echo "Postprocessing output directory: ${POST_PATH}"
echo "RDS output directory: ${RDS_PATH}"

# Create the directories if they don't exist 
mkdir -p "${MAGECK_PATH}"
mkdir -p "${BAGEL2_PATH}"
mkdir -p "${MAGECK_LOG_PATH}"
mkdir -p "${BAGEL2_LOG_PATH}"
mkdir -p "${POST_PATH}"
mkdir -p "${RDS_PATH}"

MAGeCK output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/single_guide/01_MAGeCK
MAGeCK log directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/LOGS/MAGeCK
BAGEL2 output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/single_guide/02_BAGEL2
MAGeCK log directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/LOGS/BAGEL2
Postprocessing output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/single_guide/03_postprocessing
RDS output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/RDS/single_guide


***

## Preparing input datasets for singles analyses

As the library has single genes in both orientations (e.g. geneA|safe_targeting and safe_targeting|geneB), singles analyses for each cell line are run in triplicate (datasets):

* `A` - gene|safe_targeting and safe_targeting|safe_targeting
* `B` - safe_targeting|gene and safe_targeting|safe_targeting
* `combined` - gene|safe_targeting, safe_targeting|gene and safe_targeting|safe_targeting

There are 8772 single guide pairs in the starting library (`METADATA/libraries/paralog_library.tsv`) from which 26 were removed due to high counts (`METADATA/guides_ids_to_remove.txt`) and one guide due to low read counts in the control samples (`DATA/preprocessing/filtered_guides.txt`), leaving a total of 8772 guide pairs for analysis with MAGeCK and BAGEL2.

The first step in the singles analysis is to prepare the scaled count matrix (`DATA/preprocessing/count_matrix.scaled.tsv`) and fold change matrix (`DATA/preprocessing/lfc_matrix.scaled.tsv`) for MAGeCK and BAGEL2 respectively. An R script (`SCRIPTS/single_guide/00_prepare_libraries_and_matrices.R`) is used to produce both the input matrices (`DATA/single_guide/00_input_data`) and LSF commands (`SCRIPTS/single_guide`) for submitting jobs to run the analyses. 

In [22]:
Rscript ${REPO_PATH}/SCRIPTS/single_guide/00_prepare_libraries_and_matrices.R \
    --dir ${REPO_PATH} \
    --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.scaled.tsv \
    --fc ${REPO_PATH}/DATA/preprocessing/lfc_matrix.scaled.tsv \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --out ${REPO_PATH}/DATA/single_guide/00_input_data \
    --annotations 13 \
    --ess ${REPO_PATH}/METADATA/CEGv2.txt \
    --noness ${REPO_PATH}/METADATA/NEGv1.txt

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading LFC matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.scaled.tsv
Guides in LFC matrix: 23392
Gene pairs in LFC matrix: 683
Reading count matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.scaled.tsv
Guides in count matrix: 23392
Gene pairs in count matrix: 683
Collating cell line and sample information...
Extracting single guide ids...
Number of guides (AB): 8772
Number of guides (A): 4424
Number of guides (B): 4422
Converting singles count matrix for use with MAGeCK...
Formatting MAGeCK count matrix for: combined
Number of guides combined : 8772
Number of safe-targeting co

***

## Running MAGeCK and BAGEL2

[MAGeCK](https://sourceforge.net/p/mageck) version 0.5.9.3 and [BAGEL2](https://github.com/hart-lab/bagel) version 2.0 build 114 were run across the modified matrices using LSF. Below are examples of the raw commands (i.e. without LSF parameters) which could be used to run MAGeCK and BAGEL.

Example MAGeCK command:

```
mageck test \
    --norm-method 'none' \
    --remove-zero 'none' \
    -k ${REPO_PATH}/DATA/single_guide/00_input_data/MAGeCK.singles_library.A.tsv \
    -t 'A-375 R1,A-375 R2,A-375 R3' \
    -c 'control_mean' \
    -n '${REPO_PATH}/DATA/single_guide/01_MAGeCK/A375/A/MAGeCK'
```

Example BAGEL2 command (gene):

```
BAGEL.py bf 
    -i ${REPO_PATH}/DATA/single_guide/00_input_data/BAGEL2.singles_library.A.tsv \
    -c 'A-375 R1,A-375 R2,A-375 R3' \
    -e ${REPO_PATH}/METADATA/CEGv2.txt \
    -n ${REPO_PATH}/METADATA/NEGv1.txt \
    -o ${REPO_PATH}/DATA/single_guide/02_BAGEL2/A375/A/BAGEL2.gene.bf
```

Example BAGEL2 command (sgRNA):

```
BAGEL.py bf 
    -i ${REPO_PATH}/DATA/single_guide/00_input_data/BAGEL2.singles_library.A.tsv \
    -c 'A-375 R1,A-375 R2,A-375 R3' \
    -e ${REPO_PATH}/METADATA/CEGv2.txt \
    -n ${REPO_PATH}/METADATA/NEGv1.txt \
    -r \
    -o ${REPO_PATH}/DATA/single_guide/02_BAGEL2/A375/A/BAGEL2.sgrna.bf
```

The results for the MAGeCK and BAGEL analyses can be found in `DATA/single_guide/01_MAGeCK/[cell line][dataset]` and `DATA/single_guide/02_BAGEL2/[cell line][dataset]` respectively. Logs can be found in `LOGS/MAGeCK` and `LOGS/BAGEL2` respectively.

*** 

## Collating single guide results

In [4]:
Rscript ${REPO_PATH}/SCRIPTS/single_guide/01_collate_single_guide_results.R \
    --dir ${REPO_PATH} \
    --fc ${REPO_PATH}/DATA/preprocessing/lfc_matrix.scaled.tsv \
    --annotations 13 \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --mageck ${REPO_PATH}/DATA/single_guide/01_MAGeCK \
    --bagel ${REPO_PATH}/DATA/single_guide/02_BAGEL2 \
    --ess ${REPO_PATH}/METADATA/CEGv2.txt \
    --noness ${REPO_PATH}/METADATA/NEGv1.txt \
    --out ${REPO_PATH}/DATA/single_guide/03_postprocessing

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading LFC matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.scaled.tsv
Finding MAGeCK gene results files...
Number of MAGeCK gene results: 81
Reading and processing MAGeCK gene results files...
MAGeCK gene results collated...
Finding BAGEL2 guide results files...
Number of BAGEL2 guide results: 81
Reading and processing BAGEL2 guide results files...
BAGEL2 gene results collated...
Reading in essential and non-essential genes...
Classifying BAGEL results...
Getting ROC metrics...
Determining BAGEL2 pass/fail...
Preparing BAGEL2 ROC...
Plotting BAGEL2 ROC...
BAGEL2 ROC plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finali